In [1]:
import pandas as pd
import glob
import os
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
from math import sqrt

# Find all CSV files
file_pattern = "*.csv"
csv_files = glob.glob(file_pattern)

# Process each file and store it in a list
list_of_dataframes = []
for file_path in csv_files:
    df_temp = pd.read_csv(file_path)    
    filename = os.path.basename(file_path)
    parts = filename.split('_')
    
    # Create new columns
    df_temp['model'] = parts[0]
    df_temp['method'] = parts[1]
    
    list_of_dataframes.append(df_temp)

# Combine all DataFrames
combined_df = pd.concat(list_of_dataframes, ignore_index=True)
combined_df = combined_df[['model', 'method', 'is_correct']]

In [2]:
combination_counts = combined_df.groupby(['model', 'method']).size().reset_index(name='row_count')
combination_counts.sort_values(by=['model', 'method'])

,model,method,row_count
0,deepseek,DDPO,2570
1,deepseek,DRO,2570
2,deepseek,PAT,2570
3,deepseek,RPO,2570
4,llama2,DDPO,2570
5,llama2,DRO,2570
6,llama2,PAT,2570
7,llama2,RPO,2570
8,llama3,DDPO,2570
9,llama3,DRO,2570


In [6]:
# Create a unique ID for each prompt within its group
combined_df['prompt_id'] = combined_df.groupby(['model', 'method']).cumcount()

# Define the case type based on the prompt ID
combined_df['case_type'] = np.where(combined_df['prompt_id'] < 2070, 'Attack', 'Benign')

# Pivot table
wide_df = combined_df.pivot_table(
    index=['model', 'case_type', 'prompt_id'],
    columns='method',
    values='is_correct').reset_index()

# Separate the data into attack and benign sets
attack_df = wide_df[wide_df['case_type'] == 'Attack'].copy()
benign_df = wide_df[wide_df['case_type'] == 'Benign'].copy()

# Function to Perform McNemar's Test
def run_mcnemar_tests(data, success_condition, models_to_test, comparisons_to_test):
    results_list = []
    for model in models_to_test:
        model_data = data[data['model'] == model]
        for comparison_method in comparisons_to_test:
            ddpo_success = (model_data['DDPO'] == success_condition)
            comp_success = (model_data[comparison_method] == success_condition)
            
            contingency_table = pd.crosstab(ddpo_success, comp_success)

            # Perform McNemar's test
            test_result = mcnemar(contingency_table, exact=False, correction=True)
            p_value_str = f"< .001" if test_result.pvalue < 0.001 else f"{test_result.pvalue:.3f}"

            results_list.append({
                'Model': model.capitalize(),
                'Comparison': f"DDPO vs. {comparison_method}",
                'chi^2 statistic': f"{test_result.statistic:.2f}",
                'p-value': p_value_str})
    return pd.DataFrame(results_list)

# Run Tests
models = ['llama3', 'deepseek', 'vicuna', 'llama2', 'openchat']
comparisons = ['PAT', 'RPO', 'DRO']

# Defense against Attacks
print("McNemar's Test Results for Defense Against Attacks")
attack_defense_results_df = run_mcnemar_tests(attack_df, success_condition=True, models_to_test=models, comparisons_to_test=comparisons)
print(attack_defense_results_df.to_string(index=False))
print("\nNote: Compare p-values against the Bonferroni-corrected alpha of 0.0167.\n")

# Benign Pass Rate (BPR)
print("McNemar's Test Results for Benign Pass Rate (BPR)")
bpr_results_df = run_mcnemar_tests(benign_df, success_condition=True, models_to_test=models, comparisons_to_test=comparisons)
print(bpr_results_df.to_string(index=False))
print("\nNote: Compare p-values against the Bonferroni-corrected alpha of 0.0167.")

McNemar's Test Results for Defense Against Attacks
   Model   Comparison chi^2 statistic p-value
  Llama3 DDPO vs. PAT           78.47  < .001
  Llama3 DDPO vs. RPO           18.78  < .001
  Llama3 DDPO vs. DRO            5.78   0.016
Deepseek DDPO vs. PAT         1207.07  < .001
Deepseek DDPO vs. RPO         1182.07  < .001
Deepseek DDPO vs. DRO          811.10  < .001
  Vicuna DDPO vs. PAT          406.81  < .001
  Vicuna DDPO vs. RPO           28.99  < .001
  Vicuna DDPO vs. DRO          306.67  < .001
  Llama2 DDPO vs. PAT          110.68  < .001
  Llama2 DDPO vs. RPO           18.49  < .001
  Llama2 DDPO vs. DRO           61.85  < .001
Openchat DDPO vs. PAT         1408.02  < .001
Openchat DDPO vs. RPO         1450.03  < .001
Openchat DDPO vs. DRO         1265.17  < .001

Note: Compare p-values against the Bonferroni-corrected alpha of 0.0167.

McNemar's Test Results for Benign Pass Rate (BPR)
   Model   Comparison chi^2 statistic p-value
  Llama3 DDPO vs. PAT           92.82  < .

In [10]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2/n
    centre = p + z**2/(2*n)
    margin = z * sqrt((p*(1-p))/n + z**2/(4*n**2))
    lower = (centre - margin) / denom
    upper = (centre + margin) / denom
    return (max(0, lower), min(1, upper))

models = {
    "Llama3": {
        "asr": [6.77, 5.70, 2.56, 1.64, 0.77],
        "bpr": [94.45, 75.80, 70.60, 92.40, 97.80]},
    "Deepseek": {
        "asr": [59.52, 62.13, 57.97, 40.05, 0.39],
        "bpr": [91.45, 86.80, 97.60, 95.40, 94.60]},
    "Vicuna": {
        "asr": [35.39, 21.26, 2.95, 16.23, 0.68],
        "bpr": [90.91, 79.00, 11.80, 85.40, 95.80]},
    "Llama2": {
        "asr": [9.63, 8.74, 3.53, 6.09, 1.40],
        "bpr": [69.36, 68.40, 70.00, 76.20, 93.20]},
    "Openchat": {
        "asr": [71.20, 69.28, 71.40, 62.85, 0.97],
        "bpr": [96.45, 96.20, 89.20, 98.00, 95.60]}}

n_asr = 230 # number of ASR samples
n_bpr = 500 # number of BPR samples

defenses = ["None", "PAT", "RPO", "DRO", "DDPO"]

for model, data in models.items():
    print(f"\n{model}:")
    for i, (asr, bpr) in enumerate(zip(data["asr"], data["bpr"])):
        k_asr = round(asr/100 * n_asr)
        k_bpr = round(bpr/100 * n_bpr)
        ci_asr = wilson_ci(k_asr, n_asr)
        ci_bpr = wilson_ci(k_bpr, n_bpr)
        print(f"{defenses[i]}: ASR={asr} CI=({ci_asr[0]*100:.2f}, {ci_asr[1]*100:.2f}) | "
              f"BPR={bpr} CI=({ci_bpr[0]*100:.2f}, {ci_bpr[1]*100:.2f})")


Llama3:
None: ASR=6.77 CI=(4.33, 11.00) | BPR=94.45 CI=(92.03, 96.10)
PAT: ASR=5.7 CI=(3.33, 9.43) | BPR=75.8 CI=(71.86, 79.35)
RPO: ASR=2.56 CI=(1.20, 5.57) | BPR=70.6 CI=(66.46, 74.42)
DRO: ASR=1.64 CI=(0.68, 4.39) | BPR=92.4 CI=(89.74, 94.41)
DDPO: ASR=0.77 CI=(0.24, 3.11) | BPR=97.8 CI=(96.10, 98.77)

Deepseek:
None: ASR=59.52 CI=(53.12, 65.70) | BPR=91.45 CI=(88.62, 93.55)
PAT: ASR=62.13 CI=(55.75, 68.19) | BPR=86.8 CI=(83.55, 89.49)
RPO: ASR=57.97 CI=(51.37, 64.03) | BPR=97.6 CI=(95.85, 98.62)
DRO: ASR=40.05 CI=(33.88, 46.45) | BPR=95.4 CI=(93.19, 96.92)
DDPO: ASR=0.39 CI=(0.08, 2.42) | BPR=94.6 CI=(92.26, 96.26)

Vicuna:
None: ASR=35.39 CI=(29.33, 41.59) | BPR=90.91 CI=(88.17, 93.21)
PAT: ASR=21.26 CI=(16.51, 27.05) | BPR=79.0 CI=(75.22, 82.34)
RPO: ASR=2.95 CI=(1.48, 6.15) | BPR=11.8 CI=(9.26, 14.92)
DRO: ASR=16.23 CI=(11.90, 21.39) | BPR=85.4 CI=(82.04, 88.23)
DDPO: ASR=0.68 CI=(0.24, 3.11) | BPR=95.8 CI=(93.66, 97.24)

Llama2:
None: ASR=9.63 CI=(6.40, 14.06) | BPR=69.36 CI=(